In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "retail_demo.raw.sales_transactions"
TARGET_TABLE = "retail_demo.bronze.sales_transactions"

df_raw = spark.table(SOURCE_TABLE)

df_bronze = (
    df_raw

    # Standardize basic strings
    .withColumn(
        "sales_channel",
        F.trim(F.col("sales_channel"))
    )

    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method"))
    )

    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )

    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )

    .withColumn(
        "source_system",
        F.upper(F.trim(F.col("source_system")))
    )

    # Datatype conversion
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(18,2)")
    )

    .withColumn(
        "discount_pct",
        F.col("discount_pct").cast("decimal(10,4)")
    )

    .withColumn(
        "tax_pct",
        F.col("tax_pct").cast("decimal(10,4)")
    )

    .withColumn(
        "shipping_fee",
        F.col("shipping_fee").cast("decimal(18,2)")
    )

    # Dates
    .withColumn(
        "ship_date",
        F.to_date(F.col("ship_date"))
    )

    .withColumn(
        "delivery_date",
        F.to_date(F.col("delivery_date"))
    )

    # Timestamps
    .withColumn(
        "order_timestamp",
        F.to_timestamp(F.col("order_timestamp"))
    )

    .withColumn(
        "ingestion_timestamp",
        F.to_timestamp(F.col("ingestion_timestamp"))
    )

    # Bronze metadata
    .withColumn(
        "_bronze_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_bronze_load_date",
        F.current_date()
    )
)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)